# Utils - QB - FrequencyDetector

Ce notebook illustre et vérifie le comportement de la classe `FrequencyDetector`
(`tsforecast/utils/frequency/detector.py`), qui détecte la fréquence de séries
temporelles et de jeux de données (simples ou panel), en s'appuyant en priorité sur
`pandas.infer_freq` et en étendant sa détection pour les fréquences non couvertes
nativement (trimestrielle non ancrée, semi-mensuelle, journalière ouvrée...).

**Méthodes publiques testées** (`FrequencyDetector`) :
- `detect_time_series_frequency()` : détection sur une série simple à `DatetimeIndex`
- `detect_frequency()` : détection sur une série simple OU un panel (`MultiIndex`)
- `detect_dataset_frequency()` : détection sur un `DataFrame` simple ou panel
- `validate_frequency_consistency()` : cohérence d'un ensemble de fréquences détectées

**Fonctions de commodité testées** (`tsforecast/utils/frequency/utils.py`) :
- `detect_frequency()` : point d'entrée unique `Series`/`DataFrame`
- `detect_dataset_frequency()` : équivalent fonctionnel de la méthode, avec
  `check_consistency`/`consistency_mode`
- `detect_index_frequency()` : détection directement sur un `Index` (pas une `Series`)

**Hors périmètre** : les méthodes privées/auxiliaires (`_extend_infer_freq`,
`_detect_day_frequency`, `_detect_intraday_frequency`, `_detect_daily_frequency`,
`_detect_semi_monthly_frequency`, `_detect_column_frequency`,
`_detect_panel_frequencies`, `_detect_time_series_frequencies`) ne sont pas testées
directement -- elles le sont indirectement via les méthodes publiques qui les
appellent. `target_offset_for_index()` (autre fonction de `utils.py`, dédiée à
l'ancrage d'un offset cible plutôt qu'à la détection) et `FrequencyNormalizer`/
`FrequencyConverter` sont hors périmètre : déjà couverts par `frequency_normalizer.ipynb`
et `frequency_converter.ipynb`.

On s'appuie sur les jeux de données créés dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (repris tels quels ci-dessous)
pour confronter le détecteur à des cas réalistes : fréquences mixtes, délais de
publication, couverture temporelle hétérogène par entité, et fréquence de
publication qui diffère selon l'entité pour une même variable.

## 1 - Import, instanciation et jeux de données de référence

In [ ]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe et fonctions testées
from tsforecast.utils.frequency.detector import FrequencyDetector
from tsforecast.utils.frequency.utils import (
    detect_frequency,
    detect_dataset_frequency,
    detect_index_frequency,
)
from tsforecast.panel.utils import split_variable_key

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

# Instanciation du détecteur (min_observations=2 par défaut)
det = FrequencyDetector()
print('min_observations par défaut :', det.min_observations)

### 1.1 - Jeux de données de référence (repris du notebook 3)

On recrée ici, à l'identique, les deux fonctions de génération de
`3 - QB - Panel a frequences mixtes heterogene.ipynb` : un jeu de **séries
temporelles** (indicateurs macro d'un seul pays, fréquences mensuelle/
trimestrielle/annuelle, délais de publication) et un jeu de **panel** (France/
Allemagne/Italie, couverture temporelle propre à chaque pays, et fréquence de
publication des dépenses publiques annuelle pour la France/l'Italie mais
trimestrielle pour l'Allemagne). Ces deux jeux servent de fil rouge pour les
sections 4 et 7.

In [ ]:
# Fonction de création de séries temporelles (identique au notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies."""
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    df['balance_commerciale_annuelle'] = np.nan
    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


df_timeseries = create_timeseries_dataset()
print('df_timeseries :', df_timeseries.shape, '-- colonnes :', list(df_timeseries.columns))
df_timeseries.tail()

In [ ]:
# Fonction de création d'un jeu de données de panel fictif (identique au notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies."""
    np.random.seed(seed)
    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01',
            'prod_ind_start': '2018-06-01', 'depenses_frequency': 'annuelle'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01',
            'prod_ind_start': '2019-01-01', 'depenses_frequency': 'trimestrielle'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01',
            'prod_ind_start': '2019-06-01', 'depenses_frequency': 'annuelle'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)
        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base

        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]
        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()
    return df_panel


df_panel = create_panel_dataset()
print('df_panel :', df_panel.shape, '-- entités :', df_panel.index.get_level_values('country').unique().tolist())
df_panel.groupby('country').apply(lambda g: (g.index.get_level_values('date').min(), g.index.get_level_values('date').max()))

## 2 - `detect_time_series_frequency()` : détection sur une série simple

### 2.1 - Fréquences standards, détectées nativement par `pandas.infer_freq`

In [ ]:
# pandas.infer_freq gère nativement ces fréquences : le détecteur les retrouve à l'identique
exemples_standards = {
    'D  (journalière)': pd.date_range('2024-01-01', periods=10, freq='D'),
    'W  (hebdomadaire)': pd.date_range('2024-01-01', periods=10, freq='W'),
    'MS (mensuelle, début)': pd.date_range('2024-01-01', periods=10, freq='MS'),
    'QS (trimestrielle, début)': pd.date_range('2024-01-01', periods=8, freq='QS'),
    'YS (annuelle, début)': pd.date_range('2024-01-01', periods=5, freq='YS'),
    'h  (horaire)': pd.date_range('2024-01-01', periods=10, freq='h'),
    'min (minute)': pd.date_range('2024-01-01', periods=10, freq='min'),
}
for label, idx in exemples_standards.items():
    s = pd.Series(range(len(idx)), index=idx)
    print(f"{label:28s} -> {det.detect_time_series_frequency(s)!r}")

### 2.2 - Fréquences étendues : au-delà de ce que `pandas.infer_freq` sait faire nativement

`_extend_infer_freq` prend le relais quand `infer_freq` échoue (retourne `None` ou
lève une exception). Elle distingue notamment jour ouvré (`'B'`) vs jour calendaire
(`'D'`), et détecte la semi-mensualité (`'SM'`, ex. publication le 1er et le 15).

In [ ]:
# Semi-mensuel (1er et 15 de chaque mois) : pandas.infer_freq échoue, l'extension détecte 'SM'
dates_sm = sorted(
    pd.Timestamp(y, m, d) for y in [2020, 2021] for m in range(1, 13) for d in (1, 15)
)
idx_sm = pd.DatetimeIndex(dates_sm)
print('pandas.infer_freq        :', pd.infer_freq(idx_sm))
print('detect_time_series_freq  :', det.detect_time_series_frequency(pd.Series(range(len(idx_sm)), index=idx_sm)))

print()
# Jours ouvrés sur une période qui traverse au moins un week-end -> 'B'
idx_b_long = pd.bdate_range('2024-01-01', periods=30)
print('B (traverse un week-end) :', det.detect_time_series_frequency(pd.Series(range(30), index=idx_b_long)))

# Piège : jours ouvrés SANS traverser de week-end (ex. lundi-jeudi) -> indiscernable d'un 'D'
idx_b_short = pd.bdate_range('2024-01-01', periods=4)  # lundi a jeudi

print('Piège -- B sans week-end traversé :', idx_b_short.tolist())
print('  -> detect_time_series_frequency :', det.detect_time_series_frequency(pd.Series(range(4), index=idx_b_short)))

### 2.3 - `return_format` : les 4 formats de sortie

In [ ]:
idx_q = pd.date_range('2024-01-01', periods=8, freq='QS')
s_q = pd.Series(range(8), index=idx_q)
for fmt in ('base', 'with_position', 'full', 'components'):
    print(f"{fmt:15s} -> {det.detect_time_series_frequency(s_q, return_format=fmt)!r}")

### 2.4 - Gestion des `NaN` : `dropna()` automatique avant détection

In [ ]:
idx = pd.date_range('2024-01-01', periods=12, freq='MS')
valeurs = list(range(12))
valeurs[3] = np.nan  # NaN au milieu
valeurs[-1] = np.nan  # NaN en bord (simule un délai de publication)
s_nan = pd.Series(valeurs, index=idx)
print('Fréquence malgré les NaN :', det.detect_time_series_frequency(s_nan))
print('Observations non-nulles restantes :', s_nan.dropna().shape[0], '/', s_nan.shape[0])

### 2.5 - Index non trié : tri automatique avant détection

In [ ]:
idx = pd.date_range('2024-01-01', periods=10, freq='D')
idx_shuffled = idx[[3, 0, 7, 1, 9, 2, 8, 4, 6, 5]]  # ordre volontairement mélangé
s_shuffled = pd.Series(range(10), index=idx_shuffled)
print('Index en entrée est trié :', s_shuffled.index.is_monotonic_increasing)
print('Fréquence détectée malgré le désordre :', det.detect_time_series_frequency(s_shuffled))

### 2.6 - Index non-`DatetimeIndex` mais convertible (ex. chaînes de dates)

In [ ]:
# Chaines de dates convertibles : conversion automatique via pd.to_datetime, puis détection
s_str = pd.Series(range(6), index=['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01'])
print("Index d'entrée :", type(s_str.index).__name__)
print('Fréquence détectée :', det.detect_time_series_frequency(s_str))

# Chaines NON convertibles en dates -> ValueError explicite
s_invalid = pd.Series(range(3), index=['pas', 'une', 'date'])
try:
    det.detect_time_series_frequency(s_invalid)
except ValueError as e:
    print('ValueError :', e)

### 2.7 - `min_observations` : nombre minimal d'observations requis

In [ ]:
# Par défaut (min_observations=2) : une série à 1 seule observation non-nulle échoue
s_one = pd.Series([1.0], index=pd.date_range('2024-01-01', periods=1))
try:
    det.detect_time_series_frequency(s_one)
except ValueError as e:
    print('ValueError (1 obs, min=2) :', e)

# min_observations est configurable à l'instanciation
det_strict = FrequencyDetector(min_observations=5)
s_three = pd.Series(range(3), index=pd.date_range('2024-01-01', periods=3, freq='D'))
try:
    det_strict.detect_time_series_frequency(s_three)
except ValueError as e:
    print('ValueError (3 obs, min=5) :', e)

### 2.8 - Piège : seul l'index compte, jamais le contenu de la série

La détection ne regarde que les timestamps de l'index -- les valeurs de la série ne
sont jamais inspectées. Une série de texte, de booléens ou même de `NaN` reçoit donc
une fréquence "détectée" dès lors que son index est régulier.

In [ ]:
idx = pd.date_range('2024-01-01', periods=6, freq='MS')
s_texte = pd.Series(['FR'] * 6, index=idx)  # colonne d'identifiant d'entité, pas une variable temporelle
s_nan_only = pd.Series([np.nan] * 4 + [1.0, 2.0], index=idx)
print("Série de texte constant  -> fréquence 'détectée' :", det.detect_time_series_frequency(s_texte))
print('Série presque vide (2 valeurs utiles) -> fréquence :', det.detect_time_series_frequency(s_nan_only))

### 2.9 - Piège : index irrégulier -> `None` silencieux (pas d'exception)

À la différence d'un nombre d'observations insuffisant (qui lève `ValueError`), un
index régulier en nombre mais irrégulier dans ses écarts ne lève rien : la méthode
retourne simplement `None`.

In [ ]:
idx_irregulier = pd.DatetimeIndex(['2024-01-01', '2024-01-03', '2024-01-20', '2024-02-15'])
s_irr = pd.Series(range(4), index=idx_irregulier)
resultat = det.detect_time_series_frequency(s_irr)
print('Résultat sur index irrégulier :', resultat, '(pas d’exception)')
assert resultat is None

### 2.10 - Piège : avec seulement 2 observations, la détection réussit là où
`pandas.infer_freq` seul échouerait

`pandas.infer_freq` exige au moins 3 dates et lève `ValueError` en-deçà. Le `try/except`
autour de son appel dans `detect_time_series_frequency` intercepte cette erreur et
bascule sur `_extend_infer_freq`, qui se contente d'un seul écart (donc de 2 points).
Avec `min_observations=2` (la valeur par défaut), la détection aboutit donc là où
`pandas.infer_freq` seul aurait échoué.

In [ ]:
idx2 = pd.date_range('2024-01-01', periods=2, freq='D')
try:
    pd.infer_freq(idx2)
except ValueError as e:
    print('pandas.infer_freq(idx2)          -> ValueError :', e)

s2 = pd.Series([1, 2], index=idx2)
print('detect_time_series_frequency(s2) ->', det.detect_time_series_frequency(s2))

## 3 - `detect_frequency()` (méthode) : séries simples et panel (`MultiIndex`)

### 3.1 - Délégation directe pour une série simple (sans `MultiIndex`)

In [ ]:
idx = pd.date_range('2024-01-01', periods=10, freq='D')
s_simple = pd.Series(range(10), index=idx)
print('detect_frequency            :', det.detect_frequency(s_simple))
print('detect_time_series_frequency:', det.detect_time_series_frequency(s_simple))
assert det.detect_frequency(s_simple) == det.detect_time_series_frequency(s_simple)

### 3.2 - `MultiIndex` à 2 niveaux (entité, date) : dictionnaire de fréquences par entité

In [ ]:
idx_panel = pd.MultiIndex.from_arrays([
    ['A', 'A', 'A', 'B', 'B', 'B', 'B'],
    pd.date_range('2024-01-01', periods=3, freq='D').tolist() + pd.date_range('2024-01-01', periods=4, freq='MS').tolist(),
], names=['entity', 'date'])
s_panel = pd.Series(range(7), index=idx_panel)
print(det.detect_frequency(s_panel))

### 3.3 - Piège : les clés du dictionnaire sont TOUJOURS des tuples, même pour un
seul niveau d'entité

`normalize_entity_key` (`tsforecast/panel/utils.py`) transforme systématiquement la
valeur d'entité en tuple -- `'A'` devient `('A',)`, jamais une clé scalaire. C'est
documenté dans la docstring de la méthode, mais **contredit l'exemple donné dans la
docstring de la fonction `detect_frequency()` de `utils.py`** (`{'A': 'D', 'B': 'D'}`),
qui elle délègue directement à cette méthode -- voir section 6.11.

In [ ]:
resultat = det.detect_frequency(s_panel)
print('Type des clés :', {type(k) for k in resultat})
assert all(isinstance(k, tuple) for k in resultat)
print(resultat)

### 3.4 - `MultiIndex` à 3 niveaux : clé composite normalisée

In [ ]:
idx_3lvl = pd.MultiIndex.from_arrays([
    ['France', 'France', 'France', 'Allemagne', 'Allemagne', 'Allemagne'],
    ['industrie', 'industrie', 'industrie', 'industrie', 'industrie', 'industrie'],
    pd.date_range('2024-01-01', periods=3, freq='MS').tolist() * 2,
], names=['country', 'sector', 'date'])
s_3lvl = pd.Series(range(6), index=idx_3lvl)
print(det.detect_frequency(s_3lvl))

### 3.5 - `MultiIndex` à 1 seul niveau (pas de dimension panel) : `ValueError`

In [ ]:
idx_1lvl = pd.MultiIndex.from_arrays([pd.date_range('2024-01-01', periods=3)], names=['date'])
s_1lvl = pd.Series(range(3), index=idx_1lvl)
try:
    det.detect_frequency(s_1lvl)
except ValueError as e:
    print('ValueError :', e)

### 3.6 - Piège : les entités en échec de détection sont retirées SILENCIEUSEMENT
du dictionnaire

`_detect_column_frequency` avale le `ValueError` levé par une entité n'ayant pas
assez d'observations et ne l'ajoute simplement pas au dictionnaire (pas de clé à
`None`, pas d'exception propagée). Si TOUTES les entités échouent, le dictionnaire
final est vide et la méthode retourne `None` (pas `{}`).

In [ ]:
# Entité 'A' valide (3 obs), entité 'B' insuffisante (1 seule obs, min_observations=2)
idx_mixte = pd.MultiIndex.from_tuples([
    ('A', pd.Timestamp('2024-01-01')), ('A', pd.Timestamp('2024-02-01')), ('A', pd.Timestamp('2024-03-01')),
    ('B', pd.Timestamp('2024-01-01')),
], names=['entity', 'date'])
s_mixte = pd.Series(range(4), index=idx_mixte)
print("'B' insuffisante -> silencieusement absente :", det.detect_frequency(s_mixte))

# Toutes les entités insuffisantes -> None, pas {}
idx_toutes_insuffisantes = pd.MultiIndex.from_tuples([
    ('A', pd.Timestamp('2024-01-01')), ('B', pd.Timestamp('2024-01-01')),
], names=['entity', 'date'])
s_vide = pd.Series(range(2), index=idx_toutes_insuffisantes)
resultat = det.detect_frequency(s_vide)
print('Toutes insuffisantes ->', resultat)
assert resultat is None

## 4 - `detect_dataset_frequency()` (méthode) : `DataFrame` simples et panels

### 4.1 - `DataFrame` simple avec `DatetimeIndex`

In [ ]:
# Fréquences réellement mixtes : mensuelle (production, inflation, chômage), trimestrielle
# (PIB), annuelle (balance commerciale) -- telles que construites dans le notebook 3
freq_map_ts = det.detect_dataset_frequency(df_timeseries)
for col, freq in freq_map_ts.items():
    print(f"{col:32s} -> {freq}")

### 4.2 - `time_col` : date dans une colonne plutôt que dans l'index

In [ ]:
df_ts_flat = df_timeseries.reset_index()  # 'date' redevient une colonne ordinaire
freq_map_flat = det.detect_dataset_frequency(df_ts_flat, time_col='date')
assert freq_map_flat == freq_map_ts
print('OK : résultat identique avec date en colonne (time_col) ou en index')

### 4.3 - Piège : un `time_col` erroné est ignoré SILENCIEUSEMENT (pas d'exception)
-- et le `RangeIndex` restant est mal interprété comme des dates

`detect_dataset_frequency` ne fait `df.set_index(time_col)` que si
`time_col in df.columns` -- un nom de colonne inexistant ne déclenche aucune erreur,
la détection se poursuit simplement sur l'index déjà en place (ici un `RangeIndex`
`0..N-1`, puisque `df_ts_flat` vient d'un `reset_index()`). Or
`detect_time_series_frequency` tente de convertir TOUT index non-`DatetimeIndex` via
`pd.to_datetime` (section 2.6) : les entiers `0, 1, 2...` du `RangeIndex` sont alors
interprétés comme des **nanosecondes depuis l'epoch Unix** -- une conversion qui
réussit toujours, sans erreur, et produit un `DatetimeIndex` régulier à 1ns
d'écart. Résultat : TOUTES les colonnes (y compris `'date'` elle-même,
redevenue une colonne ordinaire) se voient assigner une fréquence `'ns'` bidon.

In [ ]:
freq_map_typo = det.detect_dataset_frequency(df_ts_flat, time_col='date_qui_nexiste_pas')
print(freq_map_typo)

# Démonstration isolée du mécanisme en cause
print()
print('RangeIndex(6) converti en dates :')
print(pd.to_datetime(pd.RangeIndex(6)))
assert set(freq_map_typo.values()) == {'ns'}

### 4.4 - Panel : détection automatique via `MultiIndex` (`panel_cols=None`)

On applique la détection au panel du notebook 3 : elle retrouve bien la fréquence
**hétérogène par pays** de `depenses_publiques_pib` (annuelle pour la France/l'Italie,
trimestrielle pour l'Allemagne), exactement comme construit dans les données.

In [ ]:
freq_map_panel = det.detect_dataset_frequency(df_panel)
for country in ['France', 'Allemagne', 'Italie']:
    freq_depenses = freq_map_panel[(country, 'depenses_publiques_pib')]
    print(f"depenses_publiques_pib -- {country:10s} -> {freq_depenses}")

print()
print("PIB (identique pour tous) :", {c: freq_map_panel[(c, 'pib_trimestriel')] for c in ['France', 'Allemagne', 'Italie']})

### 4.5 - Panel "à plat" : `panel_cols` explicite sur des colonnes (sans `MultiIndex`)

Quand l'entité est une colonne ordinaire (pas un niveau d'index) et que `panel_cols`
est fourni explicitement, `detect_panel_structure` ne la reconnaît PAS comme panel
"en index" (`panel_in_index=False`) : `_detect_panel_frequencies` bascule alors sur
un `groupby(panel_cols)` colonne par colonne plutôt que sur un `groupby(level=...)`.

In [ ]:
df_plat = df_panel.reset_index(level='country')  # 'country' redevient une colonne, 'date' reste en index
print(df_plat.head(3))
print()
freq_map_plat = det.detect_dataset_frequency(df_plat, panel_cols=['country'])
print({k: v for k, v in freq_map_plat.items() if k[-1] == 'depenses_publiques_pib'})

### 4.6 - Piège : une colonne d'identifiant (non numérique) reçoit une "fréquence"
si elle n'est pas déclarée en `panel_cols`

Comme en section 2.8, seul l'index compte : sur un `DataFrame` sans `MultiIndex`
et sans `panel_cols`, TOUTES les colonnes sont traitées comme des variables
temporelles -- y compris une colonne de texte identifiant l'entité.

In [ ]:
dates = pd.date_range('2024-01-01', periods=6, freq='MS')
df_piege = pd.DataFrame({'country': ['FR'] * 6, 'value': range(6)}, index=dates)
print(det.detect_dataset_frequency(df_piege))  # 'country' reçoit aussi une fréquence !

### 4.7 - Clés de panel "épissées" `(entité…, colonne)` et `split_variable_key()`

Pour un panel, les clés du dictionnaire retourné ne sont pas imbriquées
`(entité, colonne)` mais **épissées** en un seul tuple plat -- `('France',
'depenses_publiques_pib')` pour un panel à 1 niveau d'entité. `split_variable_key`
(`tsforecast/panel/utils.py`) fait l'opération inverse.

In [ ]:
cle = ('France', 'depenses_publiques_pib')
entite, colonne = split_variable_key(cle)
print(f'clé {cle} -> entité={entite}, colonne={colonne!r}')

# Pour une série temporelle simple (pas de panel), l'entité est un tuple vide
print(split_variable_key('inflation_ipc'))

## 5 - `validate_frequency_consistency()` : cohérence de fréquences

### 5.1 - Cas cohérent

In [ ]:
freq_map_ok = {'a': 'M', 'b': 'M', 'c': 'M'}
print(det.validate_frequency_consistency(freq_map_ok))

### 5.2 - Cas incohérent, `strict=True` (défaut) -> `(False, None)`

In [ ]:
freq_map_ko = {'a': 'M', 'b': 'Q', 'c': 'M'}
print(det.validate_frequency_consistency(freq_map_ko, strict=True))

### 5.3 - Cas incohérent, `strict=False` -> fréquence modale (la plus fréquente)

In [ ]:
print(det.validate_frequency_consistency(freq_map_ko, strict=False))  # 'M' apparaît 2 fois sur 3

### 5.4 - Piège : en cas d'égalité de comptage, c'est la PREMIÈRE fréquence
insérée dans le dictionnaire qui l'emporte

`max(freq_counts, key=freq_counts.get)` ne fait aucun tri explicite : à comptage
égal, Python retourne le premier maximum rencontré selon l'ordre d'insertion du
dictionnaire (garanti depuis Python 3.7). Le résultat dépend donc de l'ORDRE des
clés de `frequency_map`, pas d'une règle de priorité entre fréquences.

In [ ]:
freq_map_egalite_1 = {'a': 'D', 'b': 'B', 'c': 'M'}  # 3 fréquences, 1 occurrence chacune
freq_map_egalite_2 = {'c': 'M', 'b': 'B', 'a': 'D'}  # mêmes fréquences, ordre d'insertion inversé
print(det.validate_frequency_consistency(freq_map_egalite_1, strict=False))
print(det.validate_frequency_consistency(freq_map_egalite_2, strict=False))
print("-> même contenu, ordre d'insertion différent, résultat différent")

### 5.5 - Dictionnaire vide -> `(False, None)`

In [ ]:
print(det.validate_frequency_consistency({}))

### 5.6 - Application : cohérence de `depenses_publiques_pib` sur le panel

La fréquence de publication des dépenses publiques est annuelle pour 2 pays sur 3
(France, Italie) et trimestrielle pour l'Allemagne : `strict=True` la rejette,
`strict=False` retient la fréquence annuelle (majoritaire, 2 pays sur 3).

In [ ]:
freq_depenses_par_pays = {
    country: freq_map_panel[(country, 'depenses_publiques_pib')]
    for country in ['France', 'Allemagne', 'Italie']
}
print('Fréquences par pays :', freq_depenses_par_pays)
print('strict=True  ->', det.validate_frequency_consistency(freq_depenses_par_pays, strict=True))
print('strict=False ->', det.validate_frequency_consistency(freq_depenses_par_pays, strict=False))

## 6 - Fonctions de commodité (`tsforecast/utils/frequency/utils.py`)

### 6.1 - `detect_frequency()` : entrée `Series` -> délégation directe vers la méthode

In [ ]:
assert detect_frequency(s_panel) == det.detect_frequency(s_panel)
print('OK : detect_frequency(series) == FrequencyDetector().detect_frequency(series)')

### 6.2 - Piège : `time_col`/`panel_cols` avec une entrée `Series` lève `ValueError`

In [ ]:
try:
    detect_frequency(s_simple, time_col='date')
except ValueError as e:
    print('ValueError :', e)

### 6.3 - `check_consistency` + `consistency_mode='modal'` (délègue à `validate_frequency_consistency`)

In [ ]:
print('map                 :', det.detect_frequency(s_mixte if False else s_panel))
print("consistency modal   :", detect_frequency(s_panel, check_consistency=True, consistency_mode='modal'))
print("consistency modal, strict=False :", detect_frequency(s_panel, check_consistency=True, consistency_mode='modal', strict=False))

### 6.4 - `check_consistency` + `consistency_mode='highest'` : fréquence la plus granulaire

S'appuie sur l'ordre de `FrequencyNormalizer` (`get_frequency_order`), PAS sur un
comptage d'occurrences : la fréquence retenue est toujours la plus fine présente,
même si elle est minoritaire.

In [ ]:
idx_hl = pd.MultiIndex.from_arrays([
    ['A'] * 12 + ['B'] * 3,
    pd.date_range('2020-01-01', periods=12, freq='MS').tolist() + pd.date_range('2020-01-01', periods=3, freq='YS').tolist(),
], names=['entity', 'date'])
s_hl = pd.Series(range(15), index=idx_hl)
print('map      :', detect_frequency(s_hl))
print("highest  :", detect_frequency(s_hl, check_consistency=True, consistency_mode='highest'))
print("modal    :", detect_frequency(s_hl, check_consistency=True, consistency_mode='modal'))

### 6.5 - Piège : `consistency_mode` invalide lève une `ValueError` explicite

In [ ]:
try:
    detect_frequency(s_hl, check_consistency=True, consistency_mode='bogus')
except ValueError as e:
    print('ValueError :', e)

### 6.6 - Piège : `check_consistency=True` sur une série SANS `MultiIndex` est
silencieusement ignoré

`check_consistency` n'est appliqué que si le résultat de la détection est un
dictionnaire (donc pour une série `MultiIndex`). Sur une série simple, le résultat
est directement une fréquence scalaire : le paramètre n'a alors aucun effet, sans
avertissement.

In [ ]:
resultat = detect_frequency(s_simple, check_consistency=True)
print(resultat, '-- même résultat que sans check_consistency :', resultat == detect_frequency(s_simple))

### 6.7 - Entrée `DataFrame` : délégation complète vers `detect_dataset_frequency()` (fonction)

In [ ]:
assert detect_frequency(df_timeseries) == detect_dataset_frequency(df_timeseries)
print('OK : detect_frequency(df) == detect_dataset_frequency(df)')

### 6.8 - `detect_dataset_frequency()` (fonction) : `check_consistency` sur les colonnes d'un `DataFrame`

In [ ]:
# df_timeseries mélange volontairement M/Q/Y -> incohérent en strict, modale (M) en non-strict
print('strict=True  :', detect_dataset_frequency(df_timeseries, check_consistency=True, strict=True))
print('strict=False :', detect_dataset_frequency(df_timeseries, check_consistency=True, strict=False))
print("highest      :", detect_dataset_frequency(df_timeseries, check_consistency=True, consistency_mode='highest'))

### 6.9 - `detect_index_frequency()` : DatetimeIndex — repose sur `index.inferred_freq`
seul, PAS sur l'extension de `FrequencyDetector`

Contrairement à `FrequencyDetector.detect_time_series_frequency`, cette fonction
n'a AUCUN mécanisme de repli : elle appelle directement la propriété
`.inferred_freq` de pandas, sans `try/except`. Résultat : sur exactement les mêmes
index, les deux détections peuvent diverger.

In [ ]:
def comparer(idx, label):
    try:
        r_index = detect_index_frequency(idx)
    except Exception as e:
        r_index = f'{type(e).__name__}: {e}'
    r_detector = det.detect_time_series_frequency(pd.Series(range(len(idx)), index=idx))
    print(f'{label}')
    print(f'  detect_index_frequency()               -> {r_index}')
    print(f'  FrequencyDetector.detect_time_series_frequency() -> {r_detector}')
    print()

# Trimestrielle : pandas l'infère nativement (même résultat)
comparer(pd.date_range('2020-01-01', periods=8, freq='QS'), 'Trimestrielle (QS)')

# Semi-mensuelle : pandas.inferred_freq échoue -> None -> ValueError propagée par normalize_frequency
comparer(idx_sm, 'Semi-mensuelle (1er/15 du mois)')

# Index irrégulier : pandas.inferred_freq échoue aussi -> ValueError, alors que
# FrequencyDetector retourne silencieusement None (voir section 2.9)
comparer(idx_irregulier, 'Irrégulier')

# Seulement 2 observations : ValueError aussi, alors que FrequencyDetector réussit (section 2.10)
comparer(idx2, '2 observations seulement')

### 6.10 - `detect_index_frequency()` : `MultiIndex` — même limite, mais propagée
pour TOUT le panel dès qu'UNE SEULE entité échoue

Sur un `MultiIndex`, `detect_index_frequency` s'appelle récursivement par entité
SANS intercepter les exceptions individuelles : une seule entité à l'index
irrégulier fait échouer l'appel entier. C'est l'inverse du comportement de
`FrequencyDetector.detect_frequency()` (section 3.6), qui retire silencieusement
les entités en échec et retourne les autres.

In [ ]:
idx_mixte_index = pd.MultiIndex.from_tuples([
    ('A', pd.Timestamp('2020-01-01')), ('A', pd.Timestamp('2020-02-01')), ('A', pd.Timestamp('2020-03-01')),
    ('B', pd.Timestamp('2020-01-01')), ('B', pd.Timestamp('2020-01-03')), ('B', pd.Timestamp('2020-01-20')),
], names=['entity', 'date'])

try:
    detect_index_frequency(idx_mixte_index)
except ValueError as e:
    print("detect_index_frequency(MultiIndex)         -> ValueError :", e)

s_mixte_index = pd.Series(range(6), index=idx_mixte_index)
print("FrequencyDetector.detect_frequency(MultiIndex) ->", det.detect_frequency(s_mixte_index))

### 6.11 - Piège de documentation : l'exemple de la docstring de `detect_frequency()`
(module) est erroné

La docstring illustre `detect_frequency(series)` sur un `MultiIndex` avec le résultat
`{'A': 'D', 'B': 'D'}` -- des clés scalaires. En pratique (section 3.3), la fonction
délègue à `FrequencyDetector.detect_frequency()` qui renvoie TOUJOURS des clés-tuples
(`{('A',): 'D', ('B',): 'D'}`). L'exemple de la docstring ne correspond pas au
comportement réel -- à corriger si ce notebook sert de base à un doctest.

In [ ]:
idx_doc = pd.MultiIndex.from_arrays([
    ['A', 'A', 'A', 'B', 'B', 'B'],
    pd.date_range('2023-01-01', periods=3, freq='D').tolist() * 2
], names=['panel_id', 'date'])
s_doc = pd.Series([1, 2, 3, 4, 5, 6], index=idx_doc)
resultat = detect_frequency(s_doc)
print('Résultat réel     :', resultat)
print("Résultat documenté: {'A': 'D', 'B': 'D'}  (docstring, INEXACT)")
assert resultat != {'A': 'D', 'B': 'D'}
assert resultat == {('A',): 'D', ('B',): 'D'}

## 7 - Synthèse : application croisée sur les jeux de données du notebook 3

### 7.1 - Jeu de séries temporelles : fréquences mixtes détectées automatiquement

In [ ]:
print('=' * 70)
for col, freq in det.detect_dataset_frequency(df_timeseries, return_format='full').items():
    print(f"{col:32s} -> {freq}")

### 7.2 - Jeu de panel : hétérogénéité de couverture temporelle ET de fréquence de
publication, toutes deux correctement isolées par entité

In [ ]:
print('--- Couverture temporelle par pays (issue de l’index, indépendante des variables) ---')
for country in ['France', 'Allemagne', 'Italie']:
    dates_country = df_panel.loc[country].index
    print(f"  {country:10s} : {dates_country.min().date()} -> {dates_country.max().date()}")

print()
print("--- Fréquence de publication de 'depenses_publiques_pib', par pays ---")
for country in ['France', 'Allemagne', 'Italie']:
    print(f"  {country:10s} : {freq_map_panel[(country, 'depenses_publiques_pib')]}")

print()
print('--- Cohérence de cette variable à travers le panel ---')
print('  strict=True  :', det.validate_frequency_consistency(freq_depenses_par_pays, strict=True))
print('  strict=False :', det.validate_frequency_consistency(freq_depenses_par_pays, strict=False))

## 8 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/frequency/test_detector.py` (un fichier existe déjà et couvre une bonne
partie des cas nominaux -- les points ci-dessous sont ceux mis en évidence ici,
notamment les divergences entre `FrequencyDetector` et `detect_index_frequency`) :

- **Seul l'index compte, jamais les valeurs** (2.8, 4.6) : une colonne de texte,
  de booléens ou de `NaN` reçoit une fréquence "détectée" tant que son index est
  régulier -- aucune validation du contenu ou du dtype de la série/colonne.
- **`None` silencieux vs `ValueError`** (2.7 vs 2.9) : un nombre d'observations
  insuffisant lève `ValueError`, mais un index irrégulier (assez d'observations,
  écarts non réguliers) retourne silencieusement `None`. Deux échecs, deux
  comportements différents -- à tester explicitement côte à côte.
- **Extension au-delà de `pandas.infer_freq`** (2.2, 2.10) : `_extend_infer_freq`
  permet de détecter avec seulement 2 observations (`min_observations=2` par
  défaut) là où `pandas.infer_freq` seul exige au moins 3 dates et lèverait. Piège
  symétrique : jours ouvrés (B) qui ne traversent aucun week-end sont indiscernables
  d'un 'D' journalier classique.
- **`detect_frequency()` (méthode) : clés TOUJOURS tuples** (3.3) : même pour un
  panel à 1 seul niveau d'entité (`('A',)`, jamais `'A'`) -- cohérent avec
  `get_unique_panel_entities`/`detect_dataset_frequency`, mais **contredit
  l'exemple de la docstring de la fonction `detect_frequency()` de `utils.py`**
  (6.11) : à corriger dans le code ou à couvrir par un test de non-régression qui
  fixe le comportement réel.
- **Échecs par entité silencieusement filtrés** (3.6) : `FrequencyDetector.detect_frequency()`
  retire du dictionnaire les entités dont la détection échoue (pas de clé `None`,
  pas d'exception) ; si TOUTES échouent, retourne `None` (pas `{}`). Comportement
  identique pour `detect_dataset_frequency()` sur les colonnes/entités d'un panel.
- **`detect_index_frequency()` diverge fortement de `FrequencyDetector`** (6.9, 6.10)
  -- c'est le point le plus important à couvrir par des tests dédiés :
    - Repose uniquement sur `index.inferred_freq` (aucun repli `_extend_infer_freq`)
      : rate les fréquences semi-mensuelles et échoue sur des cas que
      `detect_time_series_frequency` gère (2 observations, certains index
      irréguliers).
    - Sur `DatetimeIndex` : un échec de détection lève `ValueError` (via
      `normalize_frequency(None, ...)`), là où `FrequencyDetector` renvoie `None`.
    - Sur `MultiIndex` : une seule entité en échec fait échouer TOUT l'appel
      (pas de filtrage silencieux comme pour `FrequencyDetector.detect_frequency()`).
  Ces trois divergences font de `detect_index_frequency()` et de
  `FrequencyDetector` deux façons NON interchangeables de détecter une fréquence
  sur les mêmes données -- à documenter clairement dans le code si ce n'est pas
  intentionnel.
- **`time_col` erroné ignoré silencieusement, et `RangeIndex` mal interprété comme
  des dates** (4.3) : si `time_col` n'existe pas dans les colonnes du `DataFrame`,
  `detect_dataset_frequency` ne lève rien et poursuit sur l'index en place. Si cet
  index est un `RangeIndex` (typiquement après un `reset_index()`), la conversion
  `pd.to_datetime` de la section 2.6 le réinterprète comme des nanosecondes depuis
  l'epoch Unix -- **sans jamais échouer** -- et assigne une fréquence `'ns'` bidon à
  TOUTES les colonnes. Un cas à couvrir explicitement : la fonction ne distingue
  jamais un index "réellement absent de sens temporel" d'un index temporel valide.
- **Panel "à plat" via `panel_cols` explicite sur colonnes** (4.5) : chemin de
  code distinct (`groupby(panel_cols)` plutôt que `groupby(level=...)`) du panel
  `MultiIndex` auto-détecté -- mérite ses propres tests, notamment pour vérifier
  que les clés produites restent comparables (`(entité, colonne)`).
- **Égalité de comptage dans `validate_frequency_consistency(strict=False)`**
  (5.4) : `max(freq_counts, key=freq_counts.get)` retourne le premier maximum
  selon l'ORDRE D'INSERTION du dictionnaire -- pas une règle de priorité entre
  fréquences. Un test doit fixer ce comportement (ou le signaler comme fragile) car
  il dépend de l'ordre de parcours du panel/des colonnes en amont.
- **`consistency_mode='highest'` ignore les comptages** (6.4) : retient la
  fréquence la plus granulaire même minoritaire (1 occurrence sur N), contrairement
  au mode `'modal'` -- à bien distinguer dans les tests des deux modes.
- **`check_consistency` silencieusement sans effet sur une `Series` sans**
  `MultiIndex`** (6.6) : le paramètre n'agit que si le résultat intermédiaire est
  un dictionnaire ; sur une série simple il est purement ignoré.
- **Application réaliste validée** (7.1, 7.2) : sur les jeux de données du
  notebook 3, `detect_dataset_frequency()` retrouve exactement les fréquences
  mixtes construites (M/Q/Y) et isole correctement l'hétérogénéité de
  `depenses_publiques_pib` par pays (annuelle FR/IT, trimestrielle DE) -- bon
  scénario end-to-end à figer en test d'intégration.